In [ ]:
# Imports & Setup
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import copy
import warnings
warnings.filterwarnings('ignore')

#  MLP architecture (same as M2/M3)
class MLP(nn.Module):
    def __init__(self, input_dim, hidden=[128, 64, 32], num_classes=5, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# Load data
train_df = pd.read_csv('../data/ocpp_app_layer/Combined/Train.csv')
test_df  = pd.read_csv('../data/ocpp_app_layer/Combined/Test.csv')

label_col    = 'label'
feature_cols = [c for c in train_df.columns
                if c != label_col and
                train_df[c].dtype in ['float64','int64','float32','int32']]
print(f"Features: {len(feature_cols)}")

le = LabelEncoder()
y_train = le.fit_transform(train_df[label_col])
y_test  = le.transform(test_df[label_col])

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values)
X_test  = scaler.transform(test_df[feature_cols].values)

# Train baseline model
def train_model(X_tr, y_tr, input_dim, epochs=50, patience=10,
                jitter_std=0.05, seed=42):
    torch.manual_seed(seed)
    m = MLP(input_dim=input_dim)
    optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    X_t = torch.tensor(X_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr, dtype=torch.long)
    best_loss, wait, best_state = float('inf'), 0, None
    m.train()
    for epoch in range(epochs):
        idx = torch.randperm(len(X_t))
        epoch_loss = 0
        for i in range(0, len(X_t), 32):
            xb = X_t[idx[i:i+32]]
            yb = y_t[idx[i:i+32]]
            xb = xb + torch.randn_like(xb) * jitter_std
            optimizer.zero_grad()
            loss = criterion(m(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if epoch_loss < best_loss:
            best_loss, wait = epoch_loss, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    m.load_state_dict(best_state)
    m.eval()
    return m

def evaluate(m, X_np, y_np):
    with torch.no_grad():
        logits = m(torch.tensor(X_np, dtype=torch.float32))
        probs  = torch.softmax(logits, dim=1).numpy()
        preds  = np.argmax(probs, axis=1)
    acc = accuracy_score(y_np, preds)
    f1  = f1_score(y_np, preds, average='macro')
    return acc, f1, preds, probs

print("Training baseline model...")
model_v1 = train_model(X_train, y_train, input_dim=X_train.shape[1])
acc, f1, _, _ = evaluate(model_v1, X_test, y_test)
print(f"Baseline — Acc={acc:.4f}  F1={f1:.4f}")
print(f"Classes: {le.classes_}")

Features: 51
Training baseline model...
Baseline — Acc=0.9992  F1=0.9992
Classes: ['cyberattack_ocpp16_doc_idtag'
 'cyberattack_ocpp16_dos_flooding_heartbeat'
 'cyberattack_ocpp16_fdi_chargingprofile'
 'cyberattack_ocpp16_unauthorized_access' 'normal']
